# CRISP-DM Telco Customer Churn Analysis

Dataset: [Telco Customer Churn - Kaggle](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)

This notebook follows the **CRISP-DM** methodology end-to-end for a churn classification problem.

## Phase Map
- **Business Understanding**
- **Data Understanding**
- **Data Preparation**
- **Modeling**
- **Evaluation**
- **Deployment**

## Google Colab Setup

**Running this notebook in Google Colab?** This cell will automatically:
- Detect the Colab environment
- Install required packages
- Set up the correct file paths

**Running locally?** This cell will be skipped automatically.

In [ ]:
import sys

# Check if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("📍 Running in Google Colab")
    print("\n📦 Installing required packages...")
    
    # Install packages not pre-installed in Colab
    !pip install -q xgboost lightgbm shap
    
    print("✅ Packages installed successfully!")
    print("\n📂 Next step: Upload dataset or use Kaggle API")
    print("   See COLAB_SETUP_GUIDE.md for three dataset loading options:")
    print("   - Option A: Direct upload from computer")
    print("   - Option B: Kaggle API download")
    print("   - Option C: Google Drive mount")
else:
    print("💻 Running locally")
    print("✅ Using local file paths and environment")

### Dataset Loading for Colab

**Choose ONE of these methods if running in Colab:**

**Option A: Direct Upload (Easiest)**
```python
from google.colab import files
uploaded = files.upload()
filename = list(uploaded.keys())[0]
```

**Option B: Kaggle API**
```python
!kaggle datasets download -d blastchar/telco-customer-churn
!unzip -q telco-customer-churn.zip
filename = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
```

**Option C: Google Drive**
```python
from google.colab import drive
drive.mount('/content/drive')
filename = '/content/drive/My Drive/DS-Methodologies/data/Telco-Customer-Churn.csv'
```

**Local users:** Skip this cell - the notebook will use the default path.

---

## 🤖 AI LEARNING CHECKPOINT #1: Business Understanding

**Pause here!** Before moving to Data Understanding, use AI to deepen your knowledge.

### What You Just Learned
✅ How to define business objectives and success criteria  
✅ The importance of stakeholder alignment before touching data  
✅ How CRISP-DM starts with business context, not algorithms  
✅ Setting measurable goals (Recall ≥ 0.80)  

### Ask AI to Critique

Copy this prompt into **ChatGPT** or **Claude** along with the code from cell-4:

```
You are a world-renowned CRISP-DM methodology expert who has led dozens of 
Fortune 500 data science projects. Review my Business Understanding phase 
for a telco customer churn prediction project.

CONTEXT:
- Methodology: CRISP-DM
- Project: Telco Customer Churn Prediction
- Goal: Reduce churn through predictive retention campaigns

MY WORK:
[Paste the project_charter dictionary from cell-4 here]

CRITIQUE QUESTIONS:
1. Have I clearly defined the business problem and objectives?
2. Are my success criteria (Recall ≥ 0.80) appropriate for this business case?
3. What stakeholders or constraints am I missing?
4. What business questions should I be asking about churn economics?
5. How can I better align technical metrics with business value?

Please provide 5-10 actionable improvements to strengthen this phase.
Focus on methodological rigor and completeness per CRISP-DM standards.
```

### Document AI Feedback

**Paste the AI's response below and note 2-3 key improvements to apply:**

*[Your notes here]*

### Apply Improvements

**If AI suggested changes, implement them in cell-4 above and re-run.**

**Example improvements you might make:**
- Add ROI calculations or cost-benefit estimates
- Include more specific business constraints
- Define customer lifetime value (LTV) metrics
- Add regulatory or ethical considerations
- Refine success criteria based on business impact

### Compare Results

**After applying improvements:**
- Re-run cell-4
- Compare before/after
- Document what changed and why

---

**Ready to continue?** Proceed to **Data Understanding** below.

**Want to learn more?** See [HOW_TO_LEARN_WITH_AI.md](../../HOW_TO_LEARN_WITH_AI.md) for detailed guidance on AI-assisted learning.

---

> Set the project context, business objectives, and success criteria before touching data.

from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Handle path differences between Colab and local
if IN_COLAB:
    # In Colab, files are typically in /content/ after upload/download
    # If you used Option A/B above, the file should be in current directory
    # If you used Option C (Drive), update the path accordingly
    try:
        # Try to use the filename from dataset loading cell
        data = pd.read_csv(filename)
        print(f"✅ Loaded dataset from Colab: {filename}")
    except NameError:
        # If filename not defined, provide instructions
        print("⚠️  Please run one of the dataset loading options above (A, B, or C)")
        print("   Then re-run this cell")
        raise FileNotFoundError("Dataset not loaded. See instructions above.")
else:
    # Local path
    raw_path = Path('../data/raw/Telco-Customer-Churn.csv')
    if not raw_path.exists():
        raise FileNotFoundError(f'Download Telco-Customer-Churn.csv into {raw_path.parent} before running.')
    data = pd.read_csv(raw_path)
    print(f"✅ Loaded dataset from local path: {raw_path}")

print(f"\n📊 Dataset shape: {data.shape[0]:,} rows × {data.shape[1]} columns")
data.head()

In [ ]:
project_charter = {
    'stakeholders': ['VP Customer Success', 'Retention Analytics Lead', 'Data Engineering'],
    'business_objectives': ['Quantify churn risk', 'Prioritize outreach campaigns'],
    'constraints': ['Data refresh monthly', 'Model must be explainable'],
    'milestones': {
        'kickoff': 'Define outcomes and KPIs',
        'baseline_model': 'First iteration with classical ML',
        'deployment_candidate': 'Pipeline validated in UAT'
    }
}
project_charter


## Data Understanding

**Inputs**
- Kaggle dataset `Telco-Customer-Churn.csv` (download via Kaggle API)
- Data dictionary from Kaggle page

**Activities**
1. Inspect schema, datatypes, and missingness.
2. Explore churn rate and categorical distributions.
3. Visualize relationships between tenure, contract type, and churn.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

raw_path = Path('../data/raw/Telco-Customer-Churn.csv')
if not raw_path.exists():
    raise FileNotFoundError('Download Telco-Customer-Churn.csv into data/raw before running.')

data = pd.read_csv(raw_path)
data.head()


In [ ]:
summary = data.describe(include='all').transpose()
summary[['count', 'unique', 'top', 'freq']].head(10)


In [ ]:
churn_rate = data['Churn'].value_counts(normalize=True)
print('Churn distribution:\n', churn_rate)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=data, x='Churn', ax=ax[0])
ax[0].set_title('Churn Distribution')

# Class imbalance visualization
churn_counts = data['Churn'].value_counts()
ax[1].pie(churn_counts, labels=['No Churn', 'Churn'], autopct='%1.1f%%', startangle=90, colors=['#2ecc71', '#e74c3c'])
ax[1].set_title('Churn Proportion')
plt.tight_layout()
plt.show()

print(f'\nClass Imbalance Ratio: {churn_counts["No"] / churn_counts["Yes"]:.2f}:1')

### Relationship Plots
These visuals support the narrative in the Medium article. Add additional plots as needed.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(14, 10))

# Tenure vs Churn
sns.histplot(data=data, x='tenure', hue='Churn', multiple='stack', ax=ax[0, 0], bins=30)
ax[0, 0].set_title('Tenure vs Churn')
ax[0, 0].set_xlabel('Tenure (months)')

# Contract Type vs Churn
sns.countplot(data=data, x='Contract', hue='Churn', ax=ax[0, 1])
ax[0, 1].set_title('Contract Type vs Churn')
ax[0, 1].tick_params(axis='x', rotation=15)

# Monthly Charges vs Churn
sns.boxplot(data=data, x='Churn', y='MonthlyCharges', ax=ax[1, 0])
ax[1, 0].set_title('Monthly Charges Distribution by Churn')

# Internet Service vs Churn
sns.countplot(data=data, x='InternetService', hue='Churn', ax=ax[1, 1])
ax[1, 1].set_title('Internet Service vs Churn')
ax[1, 1].tick_params(axis='x', rotation=15)

fig.tight_layout()
plt.show()

# Churn rate by key categories
print('\n=== Churn Rates by Key Features ===')
for col in ['Contract', 'InternetService', 'PaymentMethod']:
    print(f'\n{col}:')
    churn_by_cat = data.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean() * 100)
    print(churn_by_cat.sort_values(ascending=False))

### Correlation Analysis
Examine relationships between numeric features and churn to identify potential predictors.

---

## 🤖 AI LEARNING CHECKPOINT #3: Data Preparation

**Pause here!** You've transformed raw data into model-ready features.

### What You Just Learned
✅ Handling missing values (TotalCharges imputation)  
✅ Encoding categorical variables (One-Hot Encoding)  
✅ Scaling numeric features (StandardScaler)  
✅ Creating train/validation/test splits with stratification  
✅ Building sklearn pipelines for reproducibility  

### Ask AI to Critique

```
You are a data preprocessing expert following CRISP-DM best practices.
Review my Data Preparation phase for telco churn prediction.

CONTEXT:
- Missing data: TotalCharges had [X] missing values, filled with median
- Encoding: One-Hot Encoding for categorical features
- Scaling: StandardScaler for numeric features
- Split: 70% train / 15% validation / 15% test (stratified)

MY CODE:
[Paste preprocessing pipeline code from cell-14 and cell-15]

CRITIQUE QUESTIONS:
1. Is median imputation appropriate for TotalCharges, or should I use a different strategy?
2. Should I handle outliers before or after splitting?
3. Are there any feature engineering opportunities I'm missing?
4. Is my train/validation/test split ratio optimal?
5. Should I consider different encodings (e.g., target encoding, ordinal)?
6. Are there any data leakage risks in my pipeline?
7. What preprocessing steps might improve model performance?

Provide 5-10 actionable improvements.
```

### Document AI Feedback

**Key recommendations:**

*[Paste AI insights here]*

### Apply Improvements

**Potential enhancements:**
- Feature engineering (e.g., tenure groups, total services count)
- Different imputation strategies
- Outlier handling
- Feature selection
- Interaction features

### Compare Results

**After changes:**
- How did feature engineering impact the feature set?
- Are there new patterns to explore?

---

**Ready to continue?** Proceed to **Modeling** below.

---

In [ ]:
# Create numeric version of dataset for correlation analysis
data_numeric = data.copy()

# Convert TotalCharges to numeric (handle empty strings)
data_numeric['TotalCharges'] = pd.to_numeric(data_numeric['TotalCharges'], errors='coerce')
data_numeric['TotalCharges'] = data_numeric['TotalCharges'].fillna(data_numeric['TotalCharges'].median())

data_numeric['Churn_Binary'] = (data['Churn'] == 'Yes').astype(int)

# Convert categorical to numeric where meaningful
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    if col in data_numeric.columns:
        data_numeric[f'{col}_num'] = (data_numeric[col] == 'Yes').astype(int)

# Select numeric columns for correlation
numeric_features = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn_Binary'] + \
                   [f'{col}_num' for col in binary_cols if col in data_numeric.columns]

correlation_matrix = data_numeric[numeric_features].corr()

# Plot correlation heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Heatmap of Numeric Features', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

# Top correlations with churn
print('\n=== Features Most Correlated with Churn ===')
churn_corr = correlation_matrix['Churn_Binary'].drop('Churn_Binary').abs().sort_values(ascending=False)
print(churn_corr.head(8))

---

## 🤖 AI LEARNING CHECKPOINT #2: Data Understanding

**Pause here!** You've explored the dataset. Now deepen your analysis with AI.

### What You Just Learned
✅ How to perform initial data exploration (shape, types, distributions)  
✅ Understanding class imbalance (churn ratio)  
✅ Identifying key features correlated with churn  
✅ Creating meaningful visualizations for stakeholders  

### Ask AI to Critique

Copy this prompt into **ChatGPT** or **Claude**:

```
You are a senior data scientist specializing in CRISP-DM methodology.
Review my Data Understanding phase for telco churn prediction.

CONTEXT:
- Dataset: Telco Customer Churn (7,043 customers, 21 features)
- Class distribution: [paste output from cell-8]
- Top correlated features: [paste output from correlation analysis]

MY ANALYSIS:
[Paste key findings: churn rate, top correlations, distribution insights]

CRITIQUE QUESTIONS:
1. Have I thoroughly explored all important aspects of the data?
2. Are there any critical patterns or anomalies I might have missed?
3. What additional EDA visualizations would strengthen my understanding?
4. How should I interpret the class imbalance for modeling?
5. What business insights can I extract from these correlations?
6. Are there potential data quality issues I should investigate?

Provide 5-10 actionable improvements for this phase.
```

### Document AI Feedback

**Key insights from AI:**

*[Paste AI response highlights here]*

### Apply Improvements

**Suggested actions to try:**
- Create additional visualizations (e.g., feature interactions)
- Investigate outliers or missing patterns more deeply
- Segment analysis by customer type
- Time-based analysis if temporal data exists
- Document data quality concerns

### Compare Results

**After improvements:**
- What new patterns did you discover?
- How will these insights influence data preparation?

---

**Ready to continue?** Proceed to **Data Preparation** below.

---

## Data Preparation

**Plan**
- Convert `TotalCharges` to numeric.
- Impute or drop missing tenure values.
- Encode categorical features using One-Hot Encoding.
- Standardize continuous features.
- Split data into train/validation/test respecting class balance.

In [ ]:
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')
missing_total = data['TotalCharges'].isna().sum()
print(f'TotalCharges missing: {missing_total}')
data['TotalCharges'] = data['TotalCharges'].fillna(data['TotalCharges'].median())


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

target = 'Churn'
X = data.drop(columns=[target, 'customerID'])
y = data[target].map({'No': 0, 'Yes': 1})
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X.select_dtypes(exclude=['object']).columns.tolist()

preprocess = ColumnTransformer(transformers=[
    ('cat', Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_cols),
    ('num', Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numeric_cols)
])

X_train, X_temp, y_train, y_temp = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)
X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, stratify=y_temp, test_size=0.5, random_state=42)
X_train.shape, X_valid.shape, X_test.shape


## Modeling

Start with an interpretable baseline (Logistic Regression) and iterate with ensemble models. Track experiments in the critique logs.

### Advanced Ensemble Models
Compare with gradient boosting algorithms (XGBoost, LightGBM) for potential performance gains.

In [ ]:
# Install if needed: !pip install xgboost
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

# XGBoost Model
xgb_model = Pipeline(steps=[
    ('prep', preprocess),
    ('model', XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        scale_pos_weight=3,  # Handle class imbalance
        random_state=42,
        eval_metric='logloss'
    ))
])
xgb_model.fit(X_train, y_train)
xgb_valid_preds = xgb_model.predict(X_valid)
xgb_valid_proba = xgb_model.predict_proba(X_valid)[:, 1]
print('=== XGBoost Performance ===')
print(classification_report(y_valid, xgb_valid_preds))
print('Validation ROC-AUC:', roc_auc_score(y_valid, xgb_valid_proba))

### Model Interpretability with SHAP
Use SHAP values to explain model predictions and identify key drivers of churn.

In [ ]:
from joblib import dump
import json
from datetime import datetime

artifacts_dir = Path('../app/artifacts')
artifacts_dir.mkdir(exist_ok=True, parents=True)

# Save the champion model pipeline
model_path = artifacts_dir / 'telco_churn_pipeline.joblib'
dump(champion_pipeline, model_path)
print(f'✓ Pipeline saved to {model_path}')

# Save feature names for reference
feature_metadata = {
    'categorical_features': categorical_cols,
    'numeric_features': numeric_cols,
    'target': target
}
with open(artifacts_dir / 'feature_metadata.json', 'w') as f:
    json.dump(feature_metadata, f, indent=2)
print(f'✓ Feature metadata saved')

# Create Model Card
model_card = {
    'model_name': 'Telco Customer Churn Predictor',
    'version': '1.0.0',
    'created_date': datetime.now().isoformat(),
    'methodology': 'CRISP-DM',
    'algorithm': champion_model,
    'training_data': {
        'source': 'Kaggle - Telco Customer Churn',
        'n_samples_train': len(X_train),
        'n_samples_valid': len(X_valid),
        'n_samples_test': len(X_test),
        'class_distribution': f'{(y_train == 0).sum()} No Churn, {(y_train == 1).sum()} Churn'
    },
    'performance_metrics': {
        'test_accuracy': float(accuracy_score(y_test, test_preds)),
        'test_precision': float(precision_score(y_test, test_preds)),
        'test_recall': float(recall_score(y_test, test_preds)),
        'test_f1': float(f1_score(y_test, test_preds)),
        'test_roc_auc': float(roc_auc_score(y_test, test_proba))
    },
    'business_requirements': {
        'primary_metric': 'Recall >= 0.80',
        'objective': 'Identify at-risk customers for retention campaigns',
        'inference_latency_target': '<150ms per prediction'
    },
    'limitations': [
        'Model trained on single telecom dataset - may not generalize to other industries',
        'Class imbalance may affect precision',
        'Requires monthly retraining as customer behavior changes',
        'Does not account for seasonality or external economic factors'
    ],
    'ethical_considerations': [
        'Ensure fair treatment across demographic groups',
        'Avoid discriminatory targeting based on protected attributes',
        'Provide transparency to customers about retention offers'
    ],
    'usage': {
        'input_format': 'JSON with customer attributes',
        'output_format': 'Churn probability (0-1) and risk category',
        'deployment_endpoint': '/score'
    }
}

with open(artifacts_dir / 'model_card.json', 'w') as f:
    json.dump(model_card, f, indent=2)
print(f'✓ Model card saved to {artifacts_dir / "model_card.json"}')

print(f'\n📦 Deployment artifacts ready in {artifacts_dir}/')

### Business Impact & Cost-Benefit Analysis
Translate model performance into business value with retention economics.

In [ ]:
# Business parameters (hypothetical but realistic)
avg_customer_lifetime_value = 1200  # Average LTV per customer
retention_campaign_cost = 100  # Cost per targeted customer
retention_success_rate = 0.35  # 35% of contacted at-risk customers are retained

# Get confusion matrix values from test set
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, test_preds)
tn, fp, fn, tp = cm.ravel()

print('=== Confusion Matrix Breakdown ===')
print(f'True Negatives (Correctly predicted no churn): {tn}')
print(f'False Positives (Predicted churn, actually stayed): {fp}')
print(f'False Negatives (Predicted no churn, actually churned): {fn}')
print(f'True Positives (Correctly predicted churn): {tp}')

# Calculate business metrics
customers_targeted = tp + fp  # All predicted churners
wasted_campaigns = fp  # False alarms
missed_opportunities = fn  # Churners we didn't catch

# Revenue saved (customers we correctly identified and retained)
customers_saved = tp * retention_success_rate
revenue_saved = customers_saved * avg_customer_lifetime_value

# Costs
campaign_costs = customers_targeted * retention_campaign_cost

# Revenue lost (customers who churned despite intervention + missed customers)
customers_lost = fn + (tp * (1 - retention_success_rate))
revenue_lost = customers_lost * avg_customer_lifetime_value

# Net benefit
net_benefit = revenue_saved - campaign_costs

print(f'\n=== Business Impact Analysis ===')
print(f'Customers Targeted for Retention: {customers_targeted}')
print(f'Estimated Customers Saved: {customers_saved:.1f}')
print(f'Revenue Saved: ${revenue_saved:,.2f}')
print(f'Campaign Costs: ${campaign_costs:,.2f}')
print(f'Revenue Lost (Missed + Failed Retention): ${revenue_lost:,.2f}')
print(f'\n💰 Net Benefit: ${net_benefit:,.2f}')
print(f'📊 ROI: {(net_benefit / campaign_costs * 100):.1f}%')

# Visualize business impact
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Cost-Benefit breakdown
categories = ['Revenue\nSaved', 'Campaign\nCosts', 'Net\nBenefit']
values = [revenue_saved, -campaign_costs, net_benefit]
colors = ['#2ecc71', '#e74c3c', '#3498db']

bars = ax[0].bar(categories, values, color=colors, alpha=0.7, edgecolor='black')
ax[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax[0].set_ylabel('Amount ($)')
ax[0].set_title('Cost-Benefit Analysis', fontsize=12, fontweight='bold')
ax[0].grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax[0].text(bar.get_x() + bar.get_width()/2., height,
               f'${abs(height):,.0f}',
               ha='center', va='bottom' if height > 0 else 'top', fontweight='bold')

# Customer flow Sankey-style visualization
customer_categories = ['True\nPositives', 'False\nPositives', 'False\nNegatives', 'True\nNegatives']
customer_counts = [tp, fp, fn, tn]
category_colors = ['#2ecc71', '#f39c12', '#e74c3c', '#95a5a6']

bars2 = ax[1].barh(customer_categories, customer_counts, color=category_colors, alpha=0.7, edgecolor='black')
ax[1].set_xlabel('Number of Customers')
ax[1].set_title('Customer Classification Breakdown', fontsize=12, fontweight='bold')
ax[1].grid(axis='x', alpha=0.3)

# Add count labels
for i, (bar, count) in enumerate(zip(bars2, customer_counts)):
    ax[1].text(count + 10, i, f'{count}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

---

## 🤖 AI LEARNING CHECKPOINT #5: Evaluation

**Pause here!** You've evaluated your model on the test set.

### What You Just Learned
✅ Evaluating on held-out test data (avoiding overfitting assessment)  
✅ Creating confusion matrices and ROC/PR curves  
✅ Interpreting classification metrics for business context  
✅ Calculating business impact (cost-benefit analysis)  
✅ Assessing model performance against success criteria  

### Ask AI to Critique

```
You are a model evaluation expert with deep knowledge of CRISP-DM methodology.
Review my Evaluation phase for telco churn prediction.

CONTEXT:
- Success Criteria: Recall ≥ 0.80
- Business Goal: Maximize retention while minimizing wasted campaign spend

TEST SET RESULTS:
[Paste classification report from cell-28]
[Paste confusion matrix values: TP, TN, FP, FN]
[Paste business impact metrics from cost-benefit analysis]

CRITIQUE QUESTIONS:
1. Did I meet the business success criteria (Recall ≥ 0.80)?
2. How should I interpret the precision-recall trade-off for this use case?
3. Are there better evaluation metrics I should consider?
4. Is my cost-benefit analysis realistic and comprehensive?
5. What threshold tuning might improve business outcomes?
6. How can I better communicate results to non-technical stakeholders?
7. What additional validation should I perform (e.g., cross-validation)?
8. Are there fairness or bias concerns I should investigate?

Provide 5-10 actionable improvements.
```

### Document AI Feedback

**Key recommendations:**

*[Paste AI insights here]*

### Apply Improvements

**Potential enhancements:**
- Threshold optimization for business metrics
- Fairness analysis across customer segments
- Cross-validation for robustness
- Error analysis (why did model fail on certain cases?)
- Calibration assessment
- Stakeholder-friendly visualizations

### Compare Results

**After improvements:**
- How did threshold changes affect business metrics?
- What patterns emerged from error analysis?

---

**Ready to continue?** Proceed to **Deployment** below.

---

### Model Comparison
Systematically compare all models across key metrics to select the champion.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve

# Collect all model predictions (excluding LightGBM)
models_dict = {
    'Logistic Regression': (valid_preds, valid_proba),
    'Random Forest': (rf_valid_preds, rf_valid_proba),
    'XGBoost': (xgb_valid_preds, xgb_valid_proba)
}

# Calculate metrics for each model
comparison_results = []
for name, (preds, proba) in models_dict.items():
    comparison_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_valid, preds),
        'Precision': precision_score(y_valid, preds),
        'Recall': recall_score(y_valid, preds),
        'F1-Score': f1_score(y_valid, preds),
        'ROC-AUC': roc_auc_score(y_valid, proba)
    })

comparison_df = pd.DataFrame(comparison_results)
print('=== Model Performance Comparison ===')
print(comparison_df.to_string(index=False))

# Visualize comparison
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Metric comparison bar chart
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
comparison_df.set_index('Model')[metrics_to_plot].plot(kind='bar', ax=ax[0], width=0.8)
ax[0].set_title('Model Performance Metrics Comparison', fontsize=12, fontweight='bold')
ax[0].set_ylabel('Score')
ax[0].set_xlabel('Model')
ax[0].legend(loc='lower right')
ax[0].set_ylim(0, 1)
ax[0].grid(axis='y', alpha=0.3)
ax[0].tick_params(axis='x', rotation=45)

# ROC curves comparison
for name, (_, proba) in models_dict.items():
    fpr, tpr, _ = roc_curve(y_valid, proba)
    ax[1].plot(fpr, tpr, label=f'{name} (AUC={roc_auc_score(y_valid, proba):.3f})')

ax[1].plot([0, 1], [0, 1], 'k--', label='Random Classifier')
ax[1].set_xlabel('False Positive Rate')
ax[1].set_ylabel('True Positive Rate')
ax[1].set_title('ROC Curves Comparison', fontsize=12, fontweight='bold')
ax[1].legend(loc='lower right')
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Select champion model (highest recall as per business requirement)
champion_model = comparison_df.loc[comparison_df['Recall'].idxmax(), 'Model']
champion_idx = comparison_df['Recall'].idxmax()
print(f'\n🏆 Champion Model: {champion_model} (Highest Recall: {comparison_df["Recall"].max():.3f})')

# Get the champion model object for deployment
if champion_model == 'Logistic Regression':
    champion_pipeline = log_reg
elif champion_model == 'Random Forest':
    champion_pipeline = rf_model
elif champion_model == 'XGBoost':
    champion_pipeline = xgb_model

---

## 🤖 AI LEARNING CHECKPOINT #4: Modeling

**Pause here!** You've trained multiple models and compared their performance.

### What You Just Learned
✅ Building baseline models (Logistic Regression)  
✅ Implementing ensemble methods (Random Forest, XGBoost, LightGBM)  
✅ Handling class imbalance (class_weight, scale_pos_weight)  
✅ Comparing multiple algorithms systematically  
✅ Model explainability with SHAP values  

### Ask AI to Critique

```
You are a machine learning expert specializing in classification problems 
and CRISP-DM methodology. Review my Modeling phase.

CONTEXT:
- Problem: Binary classification (churn prediction)
- Class imbalance: [paste ratio from earlier]
- Models trained: Logistic Regression, Random Forest, XGBoost, LightGBM

MY RESULTS:
[Paste model comparison table from the output]

Champion Model: [paste champion model name and metrics]

CRITIQUE QUESTIONS:
1. Is my model selection appropriate for this business problem?
2. How well am I handling class imbalance?
3. Should I tune hyperparameters differently?
4. Are there other algorithms I should consider?
5. Is my model comparison methodology sound?
6. How can I better interpret the SHAP values for business stakeholders?
7. Are there ensemble or stacking opportunities?
8. What's the trade-off between model complexity and interpretability?

Provide 5-10 actionable improvements.
```

### Document AI Feedback

**Key insights:**

*[Paste AI recommendations here]*

### Apply Improvements

**Potential next steps:**
- Hyperparameter tuning (GridSearchCV, RandomizedSearchCV)
- Try other algorithms (CatBoost, Neural Networks)
- Ensemble stacking
- Cost-sensitive learning adjustments
- Feature importance analysis
- Model calibration

### Compare Results

**After tuning:**
- Which metrics improved?
- Did the champion model change?
- What's the business impact of the improvements?

---

**Ready to continue?** Proceed to **Evaluation** below.

---

---

## 🤖 AI LEARNING CHECKPOINT #6: Deployment

**Final checkpoint!** You've packaged your model for production.

### What You Just Learned
✅ Serializing models with joblib  
✅ Creating model cards for documentation  
✅ Defining API contracts (input/output schemas)  
✅ Preparing artifacts for production deployment  
✅ Documenting ethical considerations and limitations  

### Ask AI to Critique

```
You are a machine learning deployment expert specializing in CRISP-DM 
and production ML systems. Review my Deployment phase.

CONTEXT:
- Model: [paste champion model name]
- Deployment: FastAPI microservice
- Artifacts: Model pipeline, feature metadata, model card

MY DEPLOYMENT PLAN:
[Paste model_card content from cell-30/357nmknfnuj]
[Paste sample API payload from cell-31]

CRITIQUE QUESTIONS:
1. Is my model serialization approach production-ready?
2. What's missing from my model card?
3. Are my ethical considerations comprehensive?
4. What monitoring should I implement post-deployment?
5. How should I handle model versioning and updates?
6. What testing strategy should I use for the API?
7. Are there scalability concerns I should address?
8. What documentation would help future maintainers?
9. How should I plan for model retraining and drift detection?

Provide 5-10 actionable improvements for production deployment.
```

### Document AI Feedback

**Key recommendations:**

*[Paste AI insights here]*

### Apply Improvements

**Production readiness checklist:**
- [ ] API testing (unit tests, integration tests)
- [ ] Monitoring setup (prediction distribution, latency, errors)
- [ ] Model versioning strategy
- [ ] Retraining pipeline
- [ ] Drift detection
- [ ] A/B testing framework
- [ ] Rollback plan
- [ ] Documentation for stakeholders

### Next Steps

**After this notebook:**
1. Test the FastAPI service (see `../app/main.py`)
2. Run Docker deployment locally
3. Set up monitoring dashboards
4. Create stakeholder presentation
5. Plan model retraining schedule

---

## 🎉 Congratulations!

You've completed a full CRISP-DM cycle for telco churn prediction!

**What you've accomplished:**
✅ Defined business objectives with measurable success criteria  
✅ Explored and understood the dataset thoroughly  
✅ Prepared clean, model-ready features  
✅ Trained and compared multiple ML algorithms  
✅ Evaluated performance against business requirements  
✅ Deployed a production-ready prediction service  

**Continue learning:**
- Compare with [KDD methodology](../../kdd_credit_fraud/notebooks/) for fraud detection
- Explore [SEMMA methodology](../../semma_bank_marketing/notebooks/) for marketing optimization
- Read [HOW_TO_LEARN_WITH_AI.md](../../HOW_TO_LEARN_WITH_AI.md) for advanced techniques

---

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

log_reg = Pipeline(steps=[('prep', preprocess), ('model', LogisticRegression(max_iter=1000))])
log_reg.fit(X_train, y_train)
valid_preds = log_reg.predict(X_valid)
valid_proba = log_reg.predict_proba(X_valid)[:, 1]
print(classification_report(y_valid, valid_preds))
print('Validation ROC-AUC:', roc_auc_score(y_valid, valid_proba))


In [ ]:
rf_model = Pipeline(steps=[('prep', preprocess), ('model', RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42))])
rf_model.fit(X_train, y_train)
rf_valid_preds = rf_model.predict(X_valid)
rf_valid_proba = rf_model.predict_proba(X_valid)[:, 1]
print(classification_report(y_valid, rf_valid_preds))
print('Validation ROC-AUC:', roc_auc_score(y_valid, rf_valid_proba))


## Evaluation

Evaluate on the untouched test set and capture key charts (ROC, precision-recall, calibration).

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay, accuracy_score, precision_score, recall_score, f1_score

# Use champion model for test evaluation
test_proba = champion_pipeline.predict_proba(X_test)[:, 1]
test_preds = (test_proba >= 0.5).astype(int)
print(f'=== Test Set Evaluation - {champion_model} ===')
print(classification_report(y_test, test_preds))
print('Test ROC-AUC:', roc_auc_score(y_test, test_proba))

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ConfusionMatrixDisplay.from_predictions(y_test, test_preds, ax=ax[0])
ax[0].set_title('Confusion Matrix')
RocCurveDisplay.from_predictions(y_test, test_proba, ax=ax[1])
ax[1].set_title('ROC Curve')
PrecisionRecallDisplay.from_predictions(y_test, test_proba, ax=ax[2])
ax[2].set_title('Precision-Recall Curve')
fig.tight_layout()
plt.show()

## Deployment

Package the trained pipeline, document assumptions, and expose a scoring endpoint via FastAPI (see `/app`).

In [ ]:
# This cell is now handled by the deployment section above (cell 357nmknfnuj)
# Keeping for backward compatibility
print(f'✓ Pipeline already saved in deployment section above')

In [ ]:
sample_payload = {
    'gender': 'Female',
    'SeniorCitizen': 0,
    'Partner': 'Yes',
    'Dependents': 'No',
    'tenure': 12,
    'PhoneService': 'Yes',
    'MultipleLines': 'No',
    'InternetService': 'Fiber optic',
    'OnlineSecurity': 'No',
    'OnlineBackup': 'No',
    'DeviceProtection': 'No',
    'TechSupport': 'No',
    'StreamingTV': 'Yes',
    'StreamingMovies': 'Yes',
    'Contract': 'Month-to-month',
    'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check',
    'MonthlyCharges': 80.65,
    'TotalCharges': 1020.5
}
sample_payload


### Next Steps
- Compare with gradient boosted trees (e.g., XGBoost, LightGBM).
- Incorporate cost-sensitive threshold tuning.
- Back-test retention strategies with marketing team.

Update the prompts in `prompts/` after each phase is reviewed by GPT-5/Claude.